## Combining Multiple Tools in a Single Agent

In this notebook, we give a single agent access to **two** different tools at the same time:
1. **Bored Activity API** (via OpenAPI) -- suggests fun activities by category
2. **Microsoft Learn Search** (via MCP) -- searches Microsoft's documentation library

The agent decides which tool to use based on the question. We'll even ask a question that requires **both** tools in one answer.

### Step 1: Install Packages

We need a handful of libraries to interact with Azure Foundry, parse OpenAPI specs, and authenticate.

In [ ]:
# Install all required packages in one go
# jsonref is needed to resolve $ref pointers inside OpenAPI spec files
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity jsonref

### Step 2: Load Configuration

All sensitive values live in a `.env` file. We load them here so the rest of the notebook can reference them safely.

In [ ]:
# os lets us read environment variables
import os

# load_dotenv reads your .env file and sets the variables for this session
from dotenv import load_dotenv

# DefaultAzureCredential handles authentication automatically
from azure.identity import DefaultAzureCredential

# AIProjectClient is the main gateway to Azure Foundry
from azure.ai.projects import AIProjectClient

# PromptAgentDefinition defines how our agent behaves
# MCPTool lets us connect to a Model Context Protocol server
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

# jsonref resolves $ref pointers inside JSON files (needed for OpenAPI specs)
import jsonref

# Pull in all environment variables from the .env file
load_dotenv()

# The Foundry project URL that the SDK will call
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

# Which deployed language model the agent should use
my_model = os.getenv("MODEL_DEPLOYMENT_NAME")

### Step 3: Connect to Foundry

Create a project client that we will use for everything -- creating agents, listing connections, and more.

In [ ]:
# Initialize the Foundry client with our endpoint and credentials
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=DefaultAzureCredential(),
)

### Step 4: Create the Chat Client

The chat client wraps the OpenAI API and lets us create conversations and send messages.

In [ ]:
# This returns an OpenAI-compatible client already wired to your Foundry project
chat_client = foundry_client.get_openai_client()

### Step 5: Load the Activity API Tool (OpenAPI)

We read a local JSON file that describes the Bored Activity API using the OpenAPI standard.  
Then we wrap it in a tool definition the agent can use to search for activities by category.

In [ ]:
# Open the OpenAPI spec file and parse it, resolving any internal $ref pointers
with open("./activities_openapi.json", "r", encoding="utf-8") as spec_file:
    activity_api_spec = jsonref.loads(spec_file.read())

# Package the parsed spec into a tool dictionary the agent understands
# "type": "openapi" tells the agent this is an external API it can call
# "auth": {"type": "anonymous"} means no API key is required
activity_api_tool = {
    "type": "openapi",
    "openapi": {
        "name": "activities",
        "spec": activity_api_spec,
        "auth": {
            "type": "anonymous"
        },
    },
}

### Step 6: Set Up the MCP Tool

Now we configure the MCP tool that connects to the Microsoft Learn search server.
The Microsoft Learn MCP server is a **public endpoint**, so no project connection or
authentication is needed. For private MCP servers, you would also pass a `project_connection_id`.

In [ ]:
# Create an MCP tool that points to the Microsoft Learn search endpoint
# server_label is a friendly name you pick for this tool
# server_url is the public MCP endpoint provided by Microsoft
# require_approval="never" means the agent can call it without asking permission first
learn_mcp_tool = MCPTool(
    server_label="microsoft_learn_server",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
)

### Step 7: Create an Agent with Both Tools

This is where the magic happens. We give a single agent access to **two** different tools:  
1. The Bored Activity API (OpenAPI) -- for suggesting activities by category  
2. The Microsoft Learn Search (MCP) -- for finding documentation and tutorials  

The agent decides which tool to use based on the question.

In [ ]:
# Pick a descriptive name for this multi-tool agent
combo_agent_name = "multi-skill-assistant"

# Register the agent in Foundry with both tools attached
combo_agent = foundry_client.agents.create_version(
    agent_name=combo_agent_name,
    definition=PromptAgentDefinition(
        # The language model that powers this agent
        model=my_model,
        # High-level instructions telling the agent what it can do
        instructions="You are a helpful assistant with two skills: (1) you can suggest fun activities using the Bored Activity API, and (2) you can search Microsoft Learn documentation. Use whichever tool is appropriate for the user's question, or both if the question covers multiple topics.",
        # Pass both tools as a list -- the agent can use either or both in a single response
        tools=[activity_api_tool, learn_mcp_tool],
    ),
)

# Confirm the agent was created
print(f"Agent created -- Name: {combo_agent.name}, ID: {combo_agent.id}")

### Step 8: Open a Chat Session

Every conversation with the agent needs its own session so the agent can keep track of context.

In [ ]:
# Start a brand-new conversation thread
chat_session = chat_client.conversations.create()

# Print the session ID for debugging
print(f"Chat session opened -- ID: {chat_session.id}")

### Step 9: Test the Multi-Tool Agent

We ask a single question that requires **both** tools -- an activity suggestion and a Microsoft Learn search.  
The agent figures out which tool handles which part.

In [ ]:
# This question intentionally covers two different topics so the agent must use both tools
test_question = "I'm bored -- suggest a few DIY activities I can try, and also find me a Microsoft Learn tutorial about deploying Azure Functions."

In [ ]:
# Send the question to our multi-tool agent
result = chat_client.responses.create(
    # Attach this message to the conversation we opened earlier
    conversation=chat_session.id,
    # Tell the service which agent should handle this request
    extra_body={
        "agent": {
            "name": combo_agent_name,
            "type": "agent_reference",
        }
    },
    # The user's question
    input=test_question,
)

# Show the agent's combined answer
print(f"Agent response: {result.output_text}")